In [ ]:

def optimum_solution_elo(difficulty_Level, skill_Level_ELO, q_Matrix, depends, k_success, k_fail):
    """
    Finds the optimum solution for skill mastery using the ELO model.
    
    Parameters:
    - difficulty_Level: Array containing the difficulty level of each task.
    - skill_Level_ELO: Initial skill levels of students based on the ELO rating system.
    - q_Matrix: A binary matrix representing which skills are required for each task.
    - depends: Dependencies between skills.
    - k_success: Skill increment factor upon success.
    - k_fail: Skill decrement factor upon failure.

    Returns:
    - skill_mastery_elo: List of skill levels at each iteration.
    - xi: List of selected tasks at each step.
    """
    # Initialize the list to store skill mastery levels over time
    skill_mastery_elo = [skill_Level_ELO.tolist()]
    
    # Initialize the global best fitness value using the linear function
    global_best = linear_function(skill_Level_ELO.tolist())
    
    # Initialize lists to store the best task sequence and skill levels
    xi = []
    best_skill_levels = skill_Level_ELO
    mastery_level = 1.5  # Target mastery level for skills
    xi_best = []  # Best task sequence
    fraction_required = []  # Placeholder for additional functionality (if needed)
    # Continue updating skills until all skills reach the mastery level
    while np.any(best_skill_levels <= mastery_level):
        previous_best = best_skill_levels  # Store the previous best skill levels
        # Iterate through each task to update skill levels
        for i in range(num_tasks):
            task_id       = i
            # Calculate the probability of success for the current task
            Prob          = probability_elo(difficulty_Level[task_id], previous_best, q_Matrix[:, task_id])
            # Update skill levels based on the Elo model
            skill_task    = skill_update_elo(previous_best, q_Matrix[:, task_id], depends, k_success, k_fail, Prob)
            # Calculate the fitness of the updated skill levels
            task_fitness  = linear_function(skill_task)
            # Update the global best fitness, best skill levels, and best task sequence
            global_best, best_skill_levels, xi_best = maximize_skill(
                task_fitness, global_best, best_skill_levels, skill_task, q_Matrix[:, task_id], task_id, xi_best
            )
        # Append the best task sequence to the list
        xi.append(xi_best)
        # Recalculate probabilities for the best task sequence
        Prob              = probability_elo(difficulty_Level[xi_best[0]], previous_best, q_Matrix[:, xi_best[0]])
        best_skill_levels = np.copy(previous_best)
        
        # Determine whether to use k_success or k_fail based on a random probability check
        if np.prod(Prob) > rand.random():
            k = k_success
        else:
            k = k_fail
        # Update skill levels based on the chosen k valu
        for i in range(len(best_skill_levels)):
            if q_Matrix[i, xi_best[0]] == 0: # Skip if the task does not require this skill
                continue
            best_skill_levels[i] += k * (1 - Prob[i])  # Apply the update factor
            
        # Store the updated skill levels to the mastery list
        skill_mastery_elo.append(best_skill_levels.tolist())
    
    # Return the final skill mastery levels and the best task sequence
    return skill_mastery_elo, xi, 

In [ ]:
# Number of students to simulate
nm_Students = 1000  

# Lists to store results for all students
all_students_skill               = []  # Stores the skill progression of all students
all_students_task_allocation_elo = []  # Stores the sequence of tasks assigned to each student
Fraction_ELO                     = []  # Tracks whether students have reached mastery
mastery_level                    = 1.5  # Define the mastery level threshold

# Loop through each student
for student_Id in tqdm(range(nm_Students)):  
    # Compute the optimal skill progression and task allocation using ELO-based modeling
    optimal_skills, task_allocation_list = optimum_solution_elo(
        difficulty_Level, skill_Level_ELO, q_Matrix, depends, k_success, k_fail
    )

    # Store the student's skill progression and task allocation sequence
    all_students_skill.append(optimal_skills)
    all_students_task_allocation_elo.append(task_allocation_list)

    # Convert the last skill level record to a NumPy array for processing
    skill_levels_array = np.array(optimal_skills[-1])

    # Check if all skills exceed the mastery threshold
    if np.all(skill_levels_array > mastery_level):
        fraction_required = 1  # Student has achieved mastery
    else:
        fraction_required = 0  # Student has not achieved mastery

    # Store the mastery result for the student
    Fraction_ELO.append(fraction_required)